<a href="https://colab.research.google.com/github/mbetagonza/MentorIA-Lab/blob/main/Practica_(C%C3%B3digo_de_seguridad).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!pip install -q gradio matplotlib numpy scipy

In [9]:
import os
os.makedirs("src", exist_ok=True)

In [10]:
%%writefile src/simulation_core.py
import numpy as np

def generate_phantom(nx=128, nz=128, width_mm=40.0, depth_mm=50.0,
                     c0=1540, rho0=1000, c_inc=1540, rho_inc=1050,
                     inc_x_mm=0, inc_z_mm=25, inc_r_mm=8,
                     c_bg=1500, rho_bg=980, bg_x_mm=0, bg_z_mm=10, bg_r_mm=3):
    """
    Genera el modelo numérico de tejido (phantom) con propiedades acústicas.
    """
    x = np.linspace(-width_mm/2, width_mm/2, nx)
    z = np.linspace(0, depth_mm, nz)
    X, Z = np.meshgrid(x, z)

    # Propiedades del tejido base
    c = np.full((nz, nx), float(c0))
    rho = np.full((nz, nx), float(rho0))

    # Inclusión circular (lesión / tejido anómalo)
    dist_inc = np.sqrt((X - inc_x_mm)**2 + (Z - inc_z_mm)**2)
    mask_inc = dist_inc <= inc_r_mm
    c[mask_inc] = float(c_inc)
    rho[mask_inc] = float(rho_inc)

    # Dispersores Rayleigh aleatorios
    np.random.seed(42)
    scatterers = np.random.normal(0, 0.1, (nz, nx))

    # Impedancia Acústica Z = rho * c
    Z_map = rho * c

    return {
        "x_mm": x,
        "z_mm": z,
        "c": c,
        "rho": rho,
        "Z": Z_map,
        "scatterers": scatterers,
        "width_mm": width_mm,
        "depth_mm": depth_mm
    }

def simulate_level1_continuous(phantom, num_lines=64, num_elements=32, f0_mhz=5.0,
                               pulse_cycles=2.5, pitch_mm=0.3, element_width_mm=0.25,
                               focus_z_mm=15.0, attenuation_db=0.5, dynamic_range_db=40.0,
                               probe_type="linear"):
    """
    Simula la propagación de ultrasonido y genera la matriz de imagen en Modo B.
    """
    x_mm = phantom["x_mm"]
    z_mm = phantom["z_mm"]

    Z = phantom["Z"]
    dZ = np.zeros_like(Z)
    dZ[1:, :] = np.diff(Z, axis=0)

    # Coeficiente de reflexión e interferencia de dispersores
    R = dZ / (2 * Z + 1e-12) + phantom["scatterers"] * 0.05

    # Factor de atenuación según profundidad y frecuencia central
    depth_cm = z_mm[:, np.newaxis] / 10.0
    att_factor = 10 ** (-(attenuation_db * f0_mhz * 2 * depth_cm) / 20.0)

    # Efecto de focalización del haz
    focus_factor = np.exp(-((z_mm[:, np.newaxis] - focus_z_mm)**2) / (2 * (focus_z_mm * 0.3)**2)) + 0.2

    # Síntesis de señal RF y envolvente
    rf_image = R * att_factor * focus_factor
    envelope = np.abs(rf_image)

    # Compresión logarítmica para visualización B-Mode
    max_val = np.max(envelope) if np.max(envelope) > 0 else 1.0
    env_norm = envelope / max_val
    b_mode_db = 20 * np.log10(env_norm + 1e-6)
    b_mode_clipped = np.clip(b_mode_db, -dynamic_range_db, 0)

    return {
        "B": b_mode_clipped,
        "RF": rf_image,
        "x_mm": x_mm,
        "z_mm": z_mm
    }

Overwriting src/simulation_core.py


In [11]:
%%writefile src/plots.py
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

def plot_bmode(b_mode_data, x_mm, z_mm):
    """
    Genera el gráfico estandarizado para imágenes de Ultrasonido en Modo B.
    """
    plt.close('all')
    fig, ax = plt.subplots(figsize=(6, 4.8), dpi=100)

    extent = [x_mm[0], x_mm[-1], z_mm[-1], z_mm[0]]

    im = ax.imshow(
        b_mode_data,
        extent=extent,
        cmap='gray',
        aspect='auto',
        vmin=-40,
        vmax=0
    )

    ax.set_title("Reconstrucción Ultrasonográfica (Modo B)", fontsize=11, fontweight='bold', pad=12, color='#0f172a')
    ax.set_xlabel("Posición Lateral [mm]", fontsize=9, fontweight='500', color='#334155')
    ax.set_ylabel("Profundidad [mm]", fontsize=9, fontweight='500', color='#334155')

    ax.tick_params(colors='#334155', labelsize=8)
    for spine in ax.spines.values():
        spine.set_color('#cbd5e1')

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Escala Dinámica [dB]", fontsize=8, color='#334155')
    cbar.ax.tick_params(labelsize=8, colors='#334155')

    fig.tight_layout()
    return fig

Overwriting src/plots.py


In [12]:
%%writefile src/missions.py
def load_mission(mission_id):
    """
    Carga el contenido teórico y experimental de cada unidad didáctica.
    """
    missions = {
        "m1": {
            "id": "m1",
            "title": "Unidad 1: Impedancia y Reflectividad Acústica",
            "objective": "Demostrar que los ecos ultrasónicos se generan en las interfaces donde existe una discontinuidad de impedancia acústica (Z = ρ · c).",
            "concept": """
### Fundamentos Teóricos: Impedancia y Reflexión

La **impedancia acústica ($Z$)** es la resistencia que opone un medio al paso de la onda sonora:
$$Z = \\rho \\cdot c$$

Donde:
* $\\rho$: Densidad del medio ($\text{kg/m}^3$)
* $c$: Velocidad de propagación ($\text{m/s}$)

Cuando el haz ultrasónico incide perpendicularmente sobre la frontera entre dos medios con distintas impedancias ($Z_1$ y $Z_2$), se produce una reflexión parcial. El **coeficiente de reflexión ($R$)** está dado por:
$$R = \\frac{Z_2 - Z_1}{Z_2 + Z_1}$$

Si $Z_1 = Z_2$, entonces $R = 0$ y no se produce eco en la interfaz (transmisión total de la energía).
""",
            "workflow": [
                "1. **Predicción:** Evalúe si una inclusión con la misma densidad y velocidad que el tejido circundante generará eco.",
                "2. **Modificación:** Varíe la velocidad de la inclusión (c_inc) de 1540 m/s a 1700 m/s para introducir un salto de impedancia.",
                "3. **Observación:** Ejecute la simulación y analice la delimitación de bordes en la imagen B-Mode.",
                "4. **Justificación:** Responda la pregunta de evaluación técnica."
            ],
            "preset": {"f0_mhz": 5.0, "mu_db": 0.5, "c_inc": 1540.0, "focus_z_mm": 15.0},
            "final_question": "¿Por qué es imprescindible aplicar gel de acoplamiento acústico entre el transductor y la piel del paciente antes de un examen?",
            "keywords": ["aire", "impedancia", "reflexion", "interfaz", "transmision", "acoplamiento", "salto"]
        },
        "m2": {
            "id": "m2",
            "title": "Unidad 2: Frecuencia Central y Atenuación Acústica",
            "objective": "Analizar el compromiso físico entre la resolución axial de la imagen y la profundidad de penetración del haz en función de la frecuencia.",
            "concept": """
### Fundamentos Teóricos: Atenuación y Frecuencia

La atenuación del ultrasonido en tejidos biológicos aumenta proporcionalmente con la frecuencia central ($f_0$):
$$\\alpha = \\mu \\cdot f_0 \\cdot d$$

Donde:
* $\\alpha$: Atenuación total ($\text{dB}$)
* $\\mu$: Coeficiente de atenuación del tejido ($\text{dB/cm/MHz}$)
* $f_0$: Frecuencia central del transductor ($\text{MHz}$)
* $d$: Profundidad de recorrido ($\text{cm}$)

A frecuencias elevadas se obtiene mayor resolución espacial, pero la profundidad de penetración disminuye sensiblemente.
""",
            "workflow": [
                "1. **Configuración inicial:** Inicie la simulación con una frecuencia de 12 MHz.",
                "2. **Observación:** Note cómo la amplitud del eco cae rápidamente a profundidades mayores a 25 mm.",
                "3. **Optimización:** Reduzca la frecuencia a 3.5 MHz para recuperar señal a mayor profundidad.",
                "4. **Análisis:** Complete la justificación técnica final."
            ],
            "preset": {"f0_mhz": 12.0, "mu_db": 0.8, "c_inc": 1650.0, "focus_z_mm": 15.0},
            "final_question": "¿Qué frecuencia seleccionaría para evaluar una estructura superficial vs. una estructura profunda y cuál es su fundamento físico?",
            "keywords": ["frecuencia", "resolucion", "penetracion", "atenuacion", "profundidad", "absorcion"]
        },
        "m3": {
            "id": "m3",
            "title": "Unidad 3: Focalización y Resolución Lateral",
            "objective": "Investigar el efecto de la focalización electrónica en la anchura del haz y su impacto en la resolución lateral a distintas profundidades.",
            "concept": """
### Fundamentos Teóricos: Focalización del Haz

La **resolución lateral** es la capacidad del sistema para distinguir dos puntos contiguos en sentido perpendicular al haz. Depende directamente de la anchura del haz ($W$):
$$W \\approx \\frac{\\lambda \\cdot z_f}{D}$$

Donde $\\lambda$ es la longitud de onda, $z_f$ la profundidad focal y $D$ la apertura activa. En la zona focal ($z = z_f$), el haz alcanza su menor anchura, maximizando la definición espacial.
""",
            "workflow": [
                "1. **Ajuste focal:** Varíe la profundidad focal entre 5 mm y 35 mm.",
                "2. **Análisis:** Compare la definición lateral del contorno de la inclusión según la posición del foco.",
                "3. **Conclusión:** Responda la verificación de conceptos."
            ],
            "preset": {"f0_mhz": 5.0, "mu_db": 0.5, "c_inc": 1650.0, "focus_z_mm": 15.0},
            "final_question": "¿Cómo afecta la posición del foco a la resolución lateral y por qué se degrada la imagen fuera de la zona focal?",
            "keywords": ["foco", "ancho", "haz", "resolucion", "lateral", "convergencia", "apertura"]
        }
    }
    return missions.get(mission_id, missions["m1"])

def evaluate_answer(mission_id, student_text):
    if not student_text or len(student_text.strip()) < 15:
        return """<div style="padding: 12px; background-color: #fef2f2; border-left: 4px solid #ef4444; color: #991b1b; border-radius: 4px;">Mínimo 15 caracteres requeridos para la evaluación.</div>"""

    m = load_mission(mission_id)
    text_lower = student_text.lower()
    found_keywords = [kw for kw in m["keywords"] if kw in text_lower]

    if len(found_keywords) >= 3:
        return f"""<div style="padding: 12px; background-color: #f0fdf4; border-left: 4px solid #16a34a; color: #166534; border-radius: 4px;"><b>Respuesta Correcta:</b> Incluye conceptos clave ({", ".join(found_keywords)}).</div>"""
    elif len(found_keywords) >= 1:
        return f"""<div style="padding: 12px; background-color: #fffbeb; border-left: 4px solid #d97706; color: #92400e; border-radius: 4px;"><b>Parcialmente Correcta:</b> Aborda el tema, pero puede profundizar más.</div>"""
    else:
        return """<div style="padding: 12px; background-color: #fef2f2; border-left: 4px solid #dc2626; color: #991b1b; border-radius: 4px;"><b>Revisión Requerida:</b> La respuesta no contiene los términos físicos esperados.</div>"""

Overwriting src/missions.py


In [13]:
%%writefile src/ui_gradio.py
import gradio as gr
from src.simulation_core import generate_phantom, simulate_level1_continuous
from src.plots import plot_bmode
from src.missions import load_mission, evaluate_answer

# Creación de tema personalizado en Azul/Celeste para sobrescribir componentes nativos
blue_theme = gr.themes.Soft(
    primary_hue="sky",
    secondary_hue="blue",
    neutral_hue="slate"
).set(
    slider_color="#0284c7",
    slider_color_dark="#0284c7",
    button_primary_background_fill="#0284c7",
    button_primary_background_fill_hover="#0369a1",
    button_primary_text_color="#ffffff"
)

LABSTER_CSS = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800&display=swap');
* { font-family: 'Inter', sans-serif !important; }

body, .gradio-container { background-color: #f8fafc !important; color: #0f172a !important; }

.labster-navbar {
    display: flex;
    justify-content: space-between;
    align-items: center;
    padding: 16px 28px;
    background-color: #ffffff;
    border-bottom: 2px solid #e0f2fe;
    margin-bottom: 24px;
    box-shadow: 0 1px 3px rgba(0,0,0,0.05);
}
.labster-brand { font-size: 1.4rem; font-weight: 800; color: #0369a1; }
.labster-brand span { color: #38bdf8; }

.hero-container { text-align: center; padding: 20px; max-width: 860px; margin: 0 auto; }
.hero-title { font-size: 2rem; font-weight: 800; color: #0c4a6e; }
.hero-subtitle { font-size: 1rem; color: #475569; margin-bottom: 20px; }

.lab-card {
    background: #ffffff;
    border: 1px solid #bae6fd;
    border-radius: 12px;
    padding: 20px;
    box-shadow: 0 4px 6px -1px rgba(0,0,0,0.03);
}

/* Sobrescritura estricta para pistas de sliders y pestañas activas */
.range-slider::-webkit-slider-runnable-track { background-color: #e0f2fe !important; }
.tab-nav button.selected { border-bottom-color: #0284c7 !important; color: #0284c7 !important; }
"""

def run_simulation(m_id, f0, mu, c_inc, focus_z):
    try:
        phantom = generate_phantom(
            nx=128, nz=128, width_mm=40.0, depth_mm=50.0,
            c0=1540, rho0=1000, c_inc=float(c_inc), rho_inc=1050,
            inc_x_mm=0, inc_z_mm=25, inc_r_mm=5
        )
        res = simulate_level1_continuous(
            phantom=phantom,
            num_lines=64,
            num_elements=32,
            f0_mhz=float(f0),
            pulse_cycles=2.5,
            pitch_mm=0.3,
            element_width_mm=0.25,
            focus_z_mm=float(focus_z),
            attenuation_db=float(mu),
            dynamic_range_db=40.0,
            probe_type="linear"
        )
        return plot_bmode(res["B"], phantom["x_mm"], phantom["z_mm"])
    except Exception as e:
        print(f"Error en la simulación: {e}")
        return None

def build_gui():
    with gr.Blocks(title="MentorIA Lab", theme=blue_theme) as demo:

        # Header
        gr.HTML("""
            <div class="labster-navbar">
                <div class="labster-brand">MentorIA <span>Lab</span></div>
                <div style="color: #0369a1; font-weight: 600;">Plataforma de Laboratorios Virtuales</div>
            </div>
        """)

        with gr.Tabs() as main_tabs:
            # PESTAÑA 0: Catálogo
            with gr.TabItem("Catálogo de Laboratorios", id=0):
                gr.HTML("""
                    <div class="hero-container">
                        <div class="hero-title">Laboratorios Virtuales de Ultrasonido</div>
                        <div class="hero-subtitle">Seleccione una unidad experimental para comenzar.</div>
                    </div>
                """)
                with gr.Row():
                    with gr.Column(elem_classes=["lab-card"]):
                        gr.Markdown("### Unidad 1\n**Impedancia y Reflectividad**")
                        btn_open_m1 = gr.Button("Abrir Laboratorio", variant="primary")
                    with gr.Column(elem_classes=["lab-card"]):
                        gr.Markdown("### Unidad 2\n**Frecuencia y Atenuación**")
                        btn_open_m2 = gr.Button("Abrir Laboratorio", variant="primary")
                    with gr.Column(elem_classes=["lab-card"]):
                        gr.Markdown("### Unidad 3\n**Focalización y Resolución**")
                        btn_open_m3 = gr.Button("Abrir Laboratorio", variant="primary")

            # PESTAÑA 1: Simulador
            with gr.TabItem("Simulador de Laboratorio", id=1):
                selected_m_id = gr.State(value="m1")

                m_selector = gr.Dropdown(
                    choices=[
                        ("Unidad 1: Impedancia y Reflectividad", "m1"),
                        ("Unidad 2: Frecuencia y Penetración", "m2"),
                        ("Unidad 3: Foco y Resolución Lateral", "m3")
                    ],
                    value="m1",
                    label="Seleccionar Unidad Experimental",
                    interactive=True
                )

                m_title = gr.Markdown("## Unidad 1: Impedancia y Reflectividad Acústica")
                m_objective = gr.Markdown("**Objetivo:** Demostrar que los ecos ultrasónicos se generan en las interfaces...")

                with gr.Accordion("Fundamentos Físicos y Teoría", open=False):
                    m_concept = gr.Markdown()

                with gr.Row():
                    with gr.Column(scale=1):
                        gr.Markdown("### Parámetros del Transductor")
                        f0_slider = gr.Slider(1.0, 15.0, value=5.0, label="Frecuencia Central (f0) [MHz]")
                        mu_slider = gr.Slider(0.1, 2.0, value=0.5, label="Atenuación (μ) [dB/cm/MHz]")
                        c_inc_slider = gr.Slider(1400.0, 1800.0, value=1540.0, label="Velocidad Inclusión [m/s]")
                        focus_slider = gr.Slider(5.0, 45.0, value=15.0, label="Profundidad Focal [mm]")

                        btn_run = gr.Button("Ejecutar Simulación", variant="primary")

                    with gr.Column(scale=2):
                        plot_output = gr.Plot(label="Reconstrucción B-Mode")

                        gr.Markdown("---")
                        gr.Markdown("### Verificación de Conceptos")
                        m_question = gr.Markdown()
                        user_answer = gr.Textbox(placeholder="Escriba su respuesta...", label="Respuesta del Estudiante", lines=2)
                        btn_submit = gr.Button("Validar Respuesta", variant="primary")
                        eval_feedback = gr.HTML()

        def change_mission(mission_id):
            m = load_mission(mission_id)
            preset = m["preset"]
            return (
                mission_id,
                f"## {m['title']}",
                f"**Objetivo:** {m['objective']}",
                m["concept"],
                f"**Pregunta de Evaluación:** {m['final_question']}",
                float(preset.get("f0_mhz", 5.0)),
                float(preset.get("mu_db", 0.5)),
                float(preset.get("c_inc", 1540.0)),
                float(preset.get("focus_z_mm", 15.0)),
                "",
                ""
            )

        m_inputs_text = [
            selected_m_id, m_title, m_objective, m_concept, m_question,
            f0_slider, mu_slider, c_inc_slider, focus_slider,
            user_answer, eval_feedback
        ]

        m_selector.change(
            fn=change_mission,
            inputs=[m_selector],
            outputs=m_inputs_text
        )

        btn_open_m1.click(
            fn=lambda: ("m1", *change_mission("m1")[1:], gr.Tabs(selected=1)),
            outputs=[m_selector] + m_inputs_text[1:] + [main_tabs]
        )
        btn_open_m2.click(
            fn=lambda: ("m2", *change_mission("m2")[1:], gr.Tabs(selected=1)),
            outputs=[m_selector] + m_inputs_text[1:] + [main_tabs]
        )
        btn_open_m3.click(
            fn=lambda: ("m3", *change_mission("m3")[1:], gr.Tabs(selected=1)),
            outputs=[m_selector] + m_inputs_text[1:] + [main_tabs]
        )

        btn_run.click(
            fn=run_simulation,
            inputs=[selected_m_id, f0_slider, mu_slider, c_inc_slider, focus_slider],
            outputs=[plot_output]
        )

        btn_submit.click(
            fn=evaluate_answer,
            inputs=[selected_m_id, user_answer],
            outputs=[eval_feedback]
        )

    return demo

Overwriting src/ui_gradio.py


In [14]:
import gradio as gr
import importlib
import src.missions
import src.ui_gradio

gr.close_all()

importlib.reload(src.missions)
importlib.reload(src.ui_gradio)

app = src.ui_gradio.build_gui()
app.launch(
    inline=True,
    share=True,
    css=src.ui_gradio.LABSTER_CSS
)

Closing server running on port: 7860


/content/src/ui_gradio.py:81: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="MentorIA Lab", theme=blue_theme) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7fa242f21adaf1a1a6.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
